# Anomaly check

QC on satellite GVF vs PhenoCam GCC and NDVI: scores table, gap by veg boxplots.

**Spin-up** (`gvf_sos == 1`): the phenology fit failed and landed on DOY 1 by
accident, not because green-up really started on Jan 1. On flat, low amplitude
curves (EN, sparse shrub, evergreen) there is no clear winter to summer swing, so
it pin SOS at the first day of data. That inflates gap /
divergence vs NDVI or GCC (noise misread as signal), so spin-up sites must be
flagged and usually excluded before interpreting lag or compression.

Artifacts: `anomaly_pipeline/output/` (`metadata/` scores, `boxplot/`,
`golden_standard_ranking.csv`).

**Veg Codes** DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub

In [47]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "anomaly_pipeline":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.data_collection import (
    build_golden_ranking,
    collect_folder,
    group_summary,
    load_table,
    plot_gap_boxplot_by_veg,
    top_n,
)

# GVF text in plotting stage (drop once, reuse here)
INPUT_DIR = REPO / "plotting_pipeline" / "input"
ANOMALY_DIR = REPO / "anomaly_pipeline" / "output"
METADATA_DIR = ANOMALY_DIR / "metadata"

## MetaData table

One row per site-year: SOS/MOS/DOS/EOS for GVF, GCC, and NDVI, plus pairwise
gap / DTW / divergence. Use it to find spin-up (`gvf_sos == 1`), large land-type
offsets, and other bad fits without opening every plot.

Writes `anomaly_pipeline/output/metadata/<FOLDER>_scores.csv`.


In [48]:
FOLDER = "GBOV_2023"  # GBOV_2024 / GoldenSites_2023
LIMIT = None
SORT = "gvf_vs_ndvi_div"
TOP = 10

csv_path = collect_folder(FOLDER, INPUT_DIR, ANOMALY_DIR, limit=LIMIT)
df = load_table(csv_path)
print(csv_path, "|", len(df), "rows")
df.head()

  BART (DB, 2023): GVF-GCC div=1.75  GVF-NDVI div=1.44  GCC-NDVI div=1.55
  HARV (DB, 2023): GVF-GCC div=5.39  GVF-NDVI div=1.44  GCC-NDVI div=6.26
  KONA (AG, 2023): GVF-GCC div=1.88  GVF-NDVI div=2.19  GCC-NDVI div=0.47
  ORNL (DB, 2023): GVF-GCC div=2.89  GVF-NDVI div=2.59  GCC-NDVI div=0.81
  DELA (DB, 2023): GVF-GCC div=2.58  GVF-NDVI div=1.70  GCC-NDVI div=2.47
  TALL (EN, 2023): GVF-GCC div=2.68  GVF-NDVI div=4.10  GCC-NDVI div=1.48
  CPER (GR, 2023): GVF-GCC div=2.26  GVF-NDVI div=2.13  GCC-NDVI div=0.66
  STER (AG, 2023): GVF-GCC div=n/a  GVF-NDVI div=8.89  GCC-NDVI div=n/a
  MOAB (GR, 2023): GVF-GCC div=3.13  GVF-NDVI div=3.48  GCC-NDVI div=1.29
  JORN (GR, 2023): GVF-GCC div=5.13  GVF-NDVI div=6.02  GCC-NDVI div=7.97
  SRER (SH, 2023): GVF-GCC div=0.65  GVF-NDVI div=1.62  GCC-NDVI div=1.13
  ONAQ (SH, 2023): GVF-GCC div=1.99  GVF-NDVI div=2.70  GCC-NDVI div=4.45

Wrote 12/12 rows to /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomal

,site,roi,veg,year,gvf_sos,gvf_mos,gvf_dos,gvf_eos,gcc_sos,gcc_mos,...,ndvi_eos,gvf_vs_gcc_div,gvf_vs_gcc_gap,gvf_vs_gcc_dtw,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gcc_vs_ndvi_div,gcc_vs_ndvi_gap,gcc_vs_ndvi_dtw
0,HARV,NEON.D01.HARV.DP1.00033_DB_1000,DB,2023,101.0,172.0,221.0,334.0,2.0,2.0,...,321.0,5.391176,74.75,0.051890,1.435059,19.25,0.060059,6.263044,87.00,0.048759
1,BART,NEON.D01.BART.DP1.00033_DB_1000,DB,2023,101.0,172.0,221.0,334.0,125.0,133.0,...,308.0,1.749500,24.00,0.035214,1.442935,19.75,0.032221,1.547836,21.25,0.029979
2,SRER,NEON.D14.SRER.DP1.00033_SH_1000,SH,2023,1.0,244.0,250.0,298.0,2.0,239.0,...,343.0,0.653057,8.00,0.081629,1.624471,21.50,0.088757,1.129437,15.00,0.058008
3,DELA,NEON.D08.DELA.DP1.00033_DB_1000,DB,2023,51.0,138.0,211.0,350.0,63.0,95.0,...,357.0,2.580651,35.75,0.027079,1.702095,23.50,0.023523,2.471499,34.25,0.025071
4,CPER,NEON.D10.CPER.DP1.00033_GR_1000,GR,2023,121.0,188.0,194.0,313.0,115.0,150.0,...,282.0,2.263203,31.50,0.013203,2.127256,29.25,0.037971,0.661021,8.75,0.036021


In [49]:
cols = [
    "site", "lag", "greenup_comp", "senescence_comp", "veg", "year",
    "gvf_sos", "gcc_sos", "ndvi_sos",
    "gvf_vs_ndvi_div", "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
    "gvf_vs_gcc_div", "gcc_vs_ndvi_div",
]
cols = [c for c in cols if c in df.columns]
display(top_n(df, by=SORT, n=TOP)[cols])
display(group_summary(df, by="veg"))


,site,veg,year,gvf_sos,gcc_sos,ndvi_sos,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gvf_vs_gcc_div,gcc_vs_ndvi_div
0,HARV,DB,2023,101.0,2.0,106.0,1.435059,19.25,0.060059,5.391176,6.263044
1,BART,DB,2023,101.0,125.0,104.0,1.442935,19.75,0.032221,1.749500,1.547836
2,SRER,SH,2023,1.0,2.0,32.0,1.624471,21.50,0.088757,0.653057,1.129437
3,DELA,DB,2023,51.0,63.0,60.0,1.702095,23.50,0.023523,2.580651,2.471499
4,CPER,GR,2023,121.0,115.0,105.0,2.127256,29.25,0.037971,2.263203,0.661021
5,KONA,AG,2023,94.0,163.0,159.0,2.189103,30.25,0.028389,1.880286,0.472121
6,ORNL,DB,2023,68.0,84.0,85.0,2.588200,32.50,0.266771,2.893593,0.812527
7,ONAQ,SH,2023,85.0,79.0,2.0,2.701735,36.50,0.094593,1.987758,4.454176
8,MOAB,GR,2023,1.0,2.0,12.0,3.483270,47.75,0.072555,3.125984,1.287827
9,TALL,EN,2023,76.0,31.0,2.0,4.098530,56.25,0.080673,2.681390,1.476684


,veg,gvf_vs_gcc_div,gvf_vs_gcc_gap,gvf_vs_gcc_dtw,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gcc_vs_ndvi_div,gcc_vs_ndvi_gap,gcc_vs_ndvi_dtw
0,AG,1.880286,26.000,0.066156,5.538550,76.375000,0.083192,0.472121,6.250,0.057740
1,DB,3.153730,43.625,0.037658,1.792072,23.750000,0.095644,2.773727,37.625,0.086227
2,EN,2.681390,36.750,0.056390,4.098530,56.250000,0.080673,1.476684,19.500,0.083827
3,GR,3.507660,47.750,0.096946,3.876714,53.166667,0.079095,3.307046,45.250,0.074903
4,SH,1.320408,17.500,0.070408,2.163103,29.000000,0.091675,2.791806,37.750,0.095378


## Satellite Data Gap boxplot by veg

Distribution of `gvf_vs_ndvi_gap` by vegetation type. Spin-up sites are red
diamonds so we can see how much they inflate the apparent discrepancy
(especially EN / GR; DB barely moves).

True lag / compression examples and the DB-vs-shrub/mixed effect-size test are
in the section after golden ranking (same clean, non-spin-up pool).

Writes `anomaly_pipeline/output/boxplot/<FOLDER>_BOXPLOT.png`.


In [50]:
boxplot_path = plot_gap_boxplot_by_veg(csv_path, ANOMALY_DIR)
print(boxplot_path)

Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png


## Golden standard ranking

Drop spin-up, then rank sites by combined GVF-GCC / GVF-NDVI divergence (gap +
DTW). Closed-canopy **DB** sites are flagged as the control group: most uniform
at VIIRS scales, tightest cross-product agreement, almost no spin-up. Their
gap/DTW distribution is the irreducible baseline under ideal conditions.

Top ranks ≈ small disagreement (baseline); mid/lower ranks still include
larger offsets. The next section pulls lag/compression examples from metadata
and tests whether shrub/mixed gap exceeds the DB baseline (Cohen's d).

Needs scores under `output/metadata/`. Writes `output/golden_standard_ranking.csv`.


In [51]:
rank_path = build_golden_ranking(ANOMALY_DIR)
rank = load_table(rank_path)
rank_cols = [
    "rank", "site", "veg", "year", "source", "golden_candidate",
    "combined_div", "combined_gap", "combined_dtw",
]
rank_cols = [c for c in rank_cols if c in rank.columns]
print(rank_path, "|", len(rank), "rows |", int(rank["golden_candidate"].sum()), "DB candidates")
display(rank.head(15)[rank_cols])
display(rank.loc[rank["golden_candidate"]].head(15)[rank_cols])

Ranked 43 site-years (22 DB golden candidates); excluded spin-up=True
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv | 43 rows | 22 DB candidates


,rank,site,veg,year,source,golden_candidate,combined_div,combined_gap,combined_dtw
0,1,blackrockforest,DB,2023,GoldenSites_2023,True,0.553277,7.250,0.035420
1,2,SRER,SH,2024,GBOV_2024,False,0.708393,9.125,0.056607
2,3,robinson2,DB,2023,GoldenSites_2023,True,0.919424,12.500,0.026567
3,4,HARV,DB,2024,GBOV_2024,True,1.019032,13.750,0.036889
4,5,bigtraillake,EN,2023,GoldenSites_2023,False,1.039068,13.750,0.056925
5,6,willowcreek,DB,2023,GoldenSites_2023,True,1.039389,13.625,0.066175
6,7,BART,DB,2024,GBOV_2024,True,1.126926,15.250,0.037641
7,8,morganmonroe2,DB,2023,GoldenSites_2023,True,1.143439,15.625,0.027367
8,9,dukehw,DB,2023,GoldenSites_2023,True,1.252109,17.125,0.028894
9,10,arkansaswhitaker,AG,2023,GoldenSites_2023,False,1.342080,18.250,0.038509


,rank,site,veg,year,source,golden_candidate,combined_div,combined_gap,combined_dtw
0,1,blackrockforest,DB,2023,GoldenSites_2023,True,0.553277,7.250,0.035420
2,3,robinson2,DB,2023,GoldenSites_2023,True,0.919424,12.500,0.026567
3,4,HARV,DB,2024,GBOV_2024,True,1.019032,13.750,0.036889
5,6,willowcreek,DB,2023,GoldenSites_2023,True,1.039389,13.625,0.066175
6,7,BART,DB,2024,GBOV_2024,True,1.126926,15.250,0.037641
7,8,morganmonroe2,DB,2023,GoldenSites_2023,True,1.143439,15.625,0.027367
8,9,dukehw,DB,2023,GoldenSites_2023,True,1.252109,17.125,0.028894
10,11,DELA,DB,2024,GBOV_2024,True,1.524166,20.875,0.033095
12,13,BART,DB,2023,GBOV_2023,True,1.596217,21.875,0.033717
13,14,SCBI,DB,2023,GoldenSites_2023,True,1.657652,22.875,0.023724


## Lag, compression, and effect size vs DB baseline

Same clean pool as ranking (spin-up excluded). Two related questions in one pass:

1. **Examples from the CSVs** spotting lag/compression in
   `metadata/` (and why they are not at the top of
   `golden_standard_ranking.csv`):
   - **Lag-ish:** large `|gvf_sos − ndvi_sos|` with a plausible green-up span
     (not spin-up). 
      - Positive value: GVF SOS after NDVI (GVF later) [actual "lag"]
      - Equal to 0 same SOS day
      - Negative value: GVF SOS before NDVI (GVF earlier)
      - difference guide: ~0–15 typical | 15–40 worth a look | 40+ lag candidate.
   - **Compression-ish (`greenup_comp`):** green-up length ratio
     `(gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)` 
      - ratio = 1 -> GVF's green-up phase took exactly as many days as GCC's. No stretching, no squeezing
      - ratio < 1 -> the numerator (GVF's duration) is smaller than the denominator (GCC's duration). GVF's green-up happened in fewer days than GCC's and GVF is compressed relative to GCC.
      - ratio > 1 ->  GVF's duration is bigger. GVF took longer to go from onset to peak than GCC did, GVF is stretched relative to GCC.
   - **Senescence** (`senescence_comp`): same idea for DOS→EOS, `(gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)` (=1 same length, <1 GVF shorter/compressed, >1 GVF longer/stretched).

2. **Effect-size test** — is shrub/mixed (`SH`+`GR`+`EN`) `gvf_vs_ndvi_gap`
   larger than the DB golden-standard mean? Cohen's d + one-sided Welch t-test
   (`H1: mixed > DB`). Only *excess* beyond the DB baseline supports a
   land-cover-driven lag claim.

Caveat: small `n` for shrub/mixed means the test can be underpowered; treat a
non-significant result as "not yet demonstrated," not proof of no effect.


### GoldenSites 2023

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GoldenSites_2023**
(spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/` (not shown here).


In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Image, display
from scipy import stats

from shared.data_collection import load_all_scores, load_table

# --- shared clean pool (used by GoldenSites + GBOV cells) ---
all_scores = load_all_scores(ANOMALY_DIR)
all_scores["spin_up"] = all_scores["gvf_sos"].eq(1.0)
clean = all_scores.loc[~all_scores["spin_up"]].copy()
print(
    f"rows={len(all_scores)} | spin-up={int(all_scores['spin_up'].sum())} | "
    f"clean={len(clean)} | sources={sorted(all_scores['source'].unique())}"
)

show_cols = [
    c for c in [
        "site", "lag", "greenup_comp", "senescence_comp", "veg", "year", "source",
        "gvf_sos", "ndvi_sos", "gvf_mos", "gcc_sos", "gcc_mos",
        "gvf_dos", "gvf_eos", "gcc_dos", "gcc_eos",
        "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
    ] if c in clean.columns
]
def _lollipop(ax, labels, values, color, ref_line=None):
    """Horizontal lollipop chart (cleaner than thick bars for ranked site values)."""
    y = np.arange(len(labels))
    vals = np.asarray(values, dtype=float)
    ax.hlines(y, 0 if ref_line is None else ref_line, vals, color=color, alpha=0.55, linewidth=1.4)
    ax.scatter(vals, y, color=color, s=42, zorder=3, edgecolors="white", linewidths=0.4)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    if ref_line is not None:
        ax.axvline(ref_line, color="black", linestyle="--", linewidth=1, alpha=0.75)
    ax.grid(True, axis="x", alpha=0.3)


def plot_lag_ranked_by_veg(df: pd.DataFrame, out_png: Path, title_prefix: str, show: bool = True):
    """Lag ranked within veg as lollipops (site-level values, not a boxplot)."""
    plot_df = df.dropna(subset=["lag", "veg", "site"]).copy()
    vegs = sorted(plot_df["veg"].unique())
    n = max(len(vegs), 1)
    max_n = int(plot_df.groupby("veg").size().max()) if len(plot_df) else 4
    fig, axes = plt.subplots(
        1, n,
        figsize=(3.6 * n, max(4.5, 0.32 * max_n + 2.2)),
        sharey=False,
    )
    if n == 1:
        axes = [axes]
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(vegs), 1)))
    for ax, veg, color in zip(axes, vegs, colors):
        sub = plot_df.loc[plot_df["veg"] == veg].sort_values("lag", ascending=True)
        _lollipop(ax, list(sub["site"]), sub["lag"].values, color, ref_line=0)
        ax.set_title(f"{veg} (n={len(sub)})")
        ax.set_xlabel("lag (days)")
    fig.suptitle(f"{title_prefix} — Lag ranked within veg (lollipop)", y=1.05)
    fig.legend(
        handles=[
            Line2D([0], [0], color="none", label="lag = gvf_sos − ndvi_sos"),
            Line2D([0], [0], color="none", label="+ : GVF SOS after NDVI (GVF later)"),
            Line2D([0], [0], color="none", label="0 : same SOS day"),
            Line2D([0], [0], color="none", label="− : GVF SOS before NDVI (GVF earlier)"),
            Line2D([0], [0], color="none", label="|lag| guide: ~0–15 typical | 15–40 look | 40+ candidate"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


def plot_compression_ranked_by_veg(df: pd.DataFrame, out_png: Path, title_prefix: str, show: bool = True):
    """greenup_comp | senescence_comp as separate side-by-side lollipop panels per veg."""
    gu_color = "#4C78A8"
    sen_color = "#F58518"
    metrics = [
        ("greenup_comp", gu_color, "greenup_comp = (gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)"),
        ("senescence_comp", sen_color, "senescence_comp = (gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)"),
    ]
    vegs = sorted(df["veg"].dropna().unique())
    n_veg = max(len(vegs), 1)
    max_n = 4
    for col, _, _ in metrics:
        if col in df.columns and df[col].notna().any():
            max_n = max(max_n, int(df.dropna(subset=[col]).groupby("veg").size().max()))

    fig, axes = plt.subplots(
        n_veg, 2,
        figsize=(11, max(3.2, 0.32 * max_n + 1.4) * n_veg),
        sharex=False,
        squeeze=False,
    )
    for row, veg in enumerate(vegs):
        for col_i, (col, color, _) in enumerate(metrics):
            ax = axes[row][col_i]
            sub = df.loc[df["veg"].eq(veg)].dropna(subset=[col, "site"]).sort_values(col, ascending=True)
            if sub.empty:
                ax.set_visible(False)
                continue
            _lollipop(ax, list(sub["site"]), sub[col].values, color, ref_line=1)
            ax.set_xlabel(col)
            if col_i == 0:
                ax.set_ylabel(veg)
            ax.set_title(f"{veg} · {col} (n={len(sub)})")

    fig.suptitle(
        f"{title_prefix} — greenup_comp (left) | senescence_comp (right)",
        y=1.01,
    )
    fig.legend(
        handles=[
            Line2D([0], [0], color=gu_color, lw=6, label=metrics[0][2]),
            Line2D([0], [0], color=sen_color, lw=6, label=metrics[1][2]),
            Line2D([0], [0], color="none", label="ratio = 1 : same length as GCC"),
            Line2D([0], [0], color="none", label="ratio < 1 : GVF shorter (compressed vs GCC)"),
            Line2D([0], [0], color="none", label="ratio > 1 : GVF longer (stretched vs GCC)"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


# --- GoldenSites 2023 ---
gs = clean.loc[clean["source"].eq("GoldenSites_2023")].copy()
print(f"\nGoldenSites_2023 clean n={len(gs)}:")
display(gs[show_cols])

lag_png = ANOMALY_DIR / "lollipopPlot" / "2023_GoldenSite_Lag.png"
comp_png = ANOMALY_DIR / "lollipopPlot" / "2023_GoldenSite_Compression.png"
plot_lag_ranked_by_veg(gs, lag_png, "2023 GoldenSites", show=False)
plot_compression_ranked_by_veg(gs, comp_png, "2023 GoldenSites", show=False)

# effect size still uses full clean pool (DB vs shrub/mixed)
db_gap = clean.loc[clean["veg"].eq("DB"), "gvf_vs_ndvi_gap"].dropna()
mixed_gap = clean.loc[clean["veg"].isin(["SH", "GR", "EN"]), "gvf_vs_ndvi_gap"].dropna()

def cohens_d(a: pd.Series, b: pd.Series) -> float:
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return float("nan")
    var_p = ((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2)
    return (b.mean() - a.mean()) / np.sqrt(var_p)

d = cohens_d(db_gap, mixed_gap)
tt = stats.ttest_ind(mixed_gap, db_gap, equal_var=False, alternative="greater")

print("\nEffect size: shrub/mixed (SH+GR+EN) vs DB golden baseline (gvf_vs_ndvi_gap)")
print(f"  DB mean gap:      {db_gap.mean():.1f} days (n={len(db_gap)})")
print(f"  Shrub/mixed mean: {mixed_gap.mean():.1f} days (n={len(mixed_gap)})")
print(f"  Cohen's d:        {d:.2f}  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)")
print(f"  one-sided p:      {tt.pvalue:.3f}  (H1: mixed > DB)")
if tt.pvalue >= 0.05:
    print(
        "  -> not significant: after dropping spin-up, shrub/mixed does not show a "
        "detectable excess lag over DB (underpowered if n_mixed is small)."
    )
else:
    print(
        "  -> significant excess gap in shrub/mixed beyond the DB baseline "
        "(still check n and spin-up screening before claiming ecology)."
    )


rows=50 | spin-up=7 | clean=43 | sources=['GBOV_2023', 'GBOV_2024', 'GoldenSites_2023']

GoldenSites_2023 clean n=22:


,site,lag,greenup_comp,senescence_comp,veg,year,source,gvf_sos,ndvi_sos,gvf_mos,gcc_sos,gcc_mos,gvf_dos,gvf_eos,gcc_dos,gcc_eos,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
23,blackrockforest,-1.0,1.5417,1.0577,DB,2023,GoldenSites_2023,113.0,114.0,150.0,113.0,137.0,221.0,331.0,216.0,320.0,7.25,0.035481
24,robinson2,-1.0,2.8235,0.8311,DB,2023,GoldenSites_2023,95.0,96.0,143.0,98.0,115.0,226.0,349.0,197.0,345.0,9.00,0.027732
25,willowcreek,-4.0,3.3077,2.3600,DB,2023,GoldenSites_2023,116.0,120.0,159.0,128.0,141.0,247.0,306.0,244.0,269.0,9.75,0.024631
26,morganmonroe2,14.0,2.1000,0.6304,DB,2023,GoldenSites_2023,98.0,84.0,140.0,93.0,113.0,253.0,340.0,202.0,340.0,10.50,0.031215
27,dukehw,-5.0,2.4138,0.8803,DB,2023,GoldenSites_2023,69.0,74.0,139.0,72.0,101.0,243.0,346.0,214.0,331.0,13.00,0.029432
28,bigtraillake,11.0,1.3051,1.1039,EN,2023,GoldenSites_2023,118.0,107.0,195.0,105.0,164.0,218.0,303.0,224.0,301.0,14.50,0.062200
29,cafcookeastltar01,-4.0,5.0000,3.5625,AG,2023,GoldenSites_2023,108.0,112.0,178.0,119.0,133.0,186.0,243.0,181.0,197.0,16.25,0.023524
30,arkansaswhitaker,17.0,1.4194,1.1667,AG,2023,GoldenSites_2023,131.0,114.0,175.0,131.0,162.0,227.0,283.0,203.0,251.0,19.25,0.037298
31,SCBI,-11.0,3.8333,1.9310,DB,2023,GoldenSites_2023,78.0,89.0,147.0,87.0,105.0,237.0,349.0,261.0,319.0,19.50,0.027887
32,coweeta,-26.0,3.6400,0.6089,DB,2023,GoldenSites_2023,68.0,94.0,159.0,97.0,122.0,236.0,345.0,163.0,342.0,22.25,0.041029


Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GoldenSite_Lag.png
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GoldenSite_Compression.png

Effect size: shrub/mixed (SH+GR+EN) vs DB golden baseline (gvf_vs_ndvi_gap)
  DB mean gap:      23.5 days (n=22)
  Shrub/mixed mean: 45.6 days (n=12)
  Cohen's d:        1.21  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)
  one-sided p:      0.007  (H1: mixed > DB)
  -> significant excess gap in shrub/mixed beyond the DB baseline (still check n and spin-up screening before claiming ecology).


### GBOV 2023 and GBOV 2024

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GBOV_2023** and
**GBOV_2024** (spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/`
(not shown here).


In [53]:
for source, file_tag, title in [
    ("GBOV_2023", "2023_GBOV", "2023 GBOV"),
    ("GBOV_2024", "2024_GBOV", "2024 GBOV"),
]:
    subset = clean.loc[clean["source"].eq(source)].copy()
    print(f"\n=== {source} clean n={len(subset)} ===")
    if subset.empty:
        print(f"No clean rows for {source}; skip.")
        continue

    display(subset[show_cols])

    lag_png = ANOMALY_DIR / "lollipopPlot" / f"{file_tag}_Lag.png"
    comp_png = ANOMALY_DIR / "lollipopPlot" / f"{file_tag}_Compression.png"
    plot_lag_ranked_by_veg(subset, lag_png, title, show=False)
    plot_compression_ranked_by_veg(subset, comp_png, title, show=False)



=== GBOV_2023 clean n=10 ===


,site,lag,greenup_comp,senescence_comp,veg,year,source,gvf_sos,ndvi_sos,gvf_mos,gcc_sos,gcc_mos,gvf_dos,gvf_eos,gcc_dos,gcc_eos,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
0,HARV,NaN,NaN,NaN,DB,2023,GBOV_2023,101.0,106.0,172.0,2.0,2.0,221.0,334.0,212.0,355.0,19.25,0.060059
1,BART,NaN,NaN,NaN,DB,2023,GBOV_2023,101.0,104.0,172.0,125.0,133.0,221.0,334.0,219.0,303.0,19.75,0.032221
3,DELA,NaN,NaN,NaN,DB,2023,GBOV_2023,51.0,60.0,138.0,63.0,95.0,211.0,350.0,138.0,365.0,23.50,0.023523
4,CPER,NaN,NaN,NaN,GR,2023,GBOV_2023,121.0,105.0,188.0,115.0,150.0,194.0,313.0,161.0,264.0,29.25,0.037971
5,KONA,NaN,NaN,NaN,AG,2023,GBOV_2023,94.0,159.0,183.0,163.0,186.0,215.0,293.0,226.0,272.0,30.25,0.028389
6,ORNL,NaN,NaN,NaN,DB,2023,GBOV_2023,68.0,85.0,152.0,84.0,106.0,207.0,349.0,258.0,302.0,32.50,0.266771
7,ONAQ,NaN,NaN,NaN,SH,2023,GBOV_2023,85.0,2.0,145.0,79.0,130.0,208.0,365.0,230.0,300.0,36.50,0.094593
9,TALL,NaN,NaN,NaN,EN,2023,GBOV_2023,76.0,2.0,145.0,31.0,216.0,212.0,337.0,243.0,337.0,56.25,0.080673
10,JORN,NaN,NaN,NaN,GR,2023,GBOV_2023,60.0,2.0,158.0,2.0,272.0,194.0,339.0,272.0,365.0,82.50,0.126758
11,STER,NaN,NaN,NaN,AG,2023,GBOV_2023,81.0,2.0,172.0,NaN,NaN,176.0,231.0,NaN,NaN,122.50,0.137996


Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GBOV_Lag.png
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GBOV_Compression.png

=== GBOV_2024 clean n=11 ===


,site,lag,greenup_comp,senescence_comp,veg,year,source,gvf_sos,ndvi_sos,gvf_mos,gcc_sos,gcc_mos,gvf_dos,gvf_eos,gcc_dos,gcc_eos,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
12,SRER,15.0,0.6667,1.0513,SH,2024,GBOV_2024,39.0,24.0,79.0,19.0,79.0,79.0,366.0,92.0,365.0,9.75,0.047216
13,BART,3.0,5.7778,1.5902,DB,2024,GBOV_2024,115.0,112.0,167.0,130.0,139.0,229.0,326.0,235.0,296.0,10.75,0.044301
14,HARV,-5.0,4.3333,0.7886,DB,2024,GBOV_2024,115.0,120.0,167.0,128.0,140.0,229.0,326.0,211.0,334.0,11.00,0.035721
15,ORNL,-43.0,5.5238,1.0440,DB,2024,GBOV_2024,43.0,86.0,159.0,87.0,108.0,235.0,330.0,231.0,322.0,23.25,0.045148
16,DELA,-4.0,3.4375,0.8954,DB,2024,GBOV_2024,71.0,75.0,126.0,79.0,95.0,152.0,366.0,126.0,365.0,25.25,0.033498
17,MOAB,-33.0,0.6203,2.5385,GR,2024,GBOV_2024,31.0,64.0,178.0,2.0,239.0,239.0,305.0,247.0,273.0,31.50,0.069665
18,ONAQ,24.0,1.3521,0.6497,SH,2024,GBOV_2024,43.0,19.0,139.0,42.0,113.0,139.0,267.0,113.0,310.0,34.75,0.078202
19,KONA,15.0,1.9242,0.6242,AG,2024,GBOV_2024,35.0,20.0,162.0,42.0,108.0,221.0,314.0,128.0,277.0,52.75,0.046400
20,TALL,75.0,0.2995,2.6548,EN,2024,GBOV_2024,77.0,2.0,139.0,11.0,218.0,143.0,366.0,278.0,362.0,54.25,0.062004
21,JORN,76.0,0.3714,0.6275,GR,2024,GBOV_2024,78.0,2.0,156.0,2.0,212.0,237.0,333.0,212.0,365.0,70.25,0.130059


Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2024_GBOV_Lag.png
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2024_GBOV_Compression.png


2023 golden site Findings:
Veg codes: DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub, EB = evergreen broadleaf

1) Compression: 
    - For Agriculture lands (n=6), green-up compression value shows mostly value greater than 1 which indiate gvf being stretched relative to gcc and senescence compression value also shows mostly value greater than 1 which is also stretched.
    - For deciduous broadleaf (n=14), green-up compression value is all positive which mean stretched. for senescence compression value its half site being stretched and half site being compressed. (could also say 2-3 site is a almost exact match (no stretch or compress, value = 1))
    - For evergreen needle and Grassland (both n = 1) all of the green-up and senescence seem to say it's being stretched
2) Lag:
    - EN and GR lag is insignificant 
    - For Agriculture lands (n=6), the lag various between -50 day eariler vs 50 day later SOS
    - For deciduous broadleaf (n=14), most site is a few days eariler but a few site are 50 days later SOS
